# Credit Risk Prediction - Project Hub

This notebook provides a summary of all results. Run the analysis notebooks in the order listed below to reproduce the full pipeline.

Key results:
- Test AUC-ROC: 0.888 (MLP), 0.893 (XGBoost)
- Fairness: disparate impact ratio 1.021 across gender groups
- Counterfactuals: 100 percent flip rate on high-risk cases, average 2.2 features changed

## Analysis Notebooks

Run in the following order:

1. [data_cleaning.ipynb](notebooks/data_cleaning.ipynb)
2. [feature_engineering.ipynb](notebooks/feature_engineering.ipynb)
3. [EDA.ipynb](notebooks/EDA.ipynb)
4. [mlp_training.ipynb](notebooks/mlp_training.ipynb)
5. [xgboost_training.ipynb](notebooks/xgboost_training.ipynb)
6. [model_evaluation.ipynb](notebooks/model_evaluation.ipynb)
7. [bias_fairness_analysis.ipynb](notebooks/bias_fairness_analysis.ipynb)
8. [generate_counterfactuals.ipynb](notebooks/generate_counterfactuals.ipynb)
9. [counterfactual_summary_statistics.ipynb](notebooks/counterfactual_summary_statistics.ipynb)

## Setup and Load Results

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import joblib
from pathlib import Path

## 1. Model Performance Summary

In [2]:
# Load metrics
with open('results/mlp_metrics.json', 'r') as f:
    metrics = json.load(f)

# Load predictions
preds = pd.read_csv('results/mlp_predictions.csv')

print("=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"\nArchitecture: {metrics['architecture']}")
print(f"Loss Function: {metrics['loss_function']}")

print(f"\nTest Set Metrics:")
print(f"  AUC-ROC:     {metrics['test_metrics']['auc_roc']:.4f}")
print(f"  AUC-PR:      {metrics['test_metrics']['auc_pr']:.4f}")
print(f"  Brier Score: {metrics['test_metrics']['brier_score']:.4f}")

print("\n" + "=" * 70)

MODEL PERFORMANCE

Architecture: 67 → 128 → 64 → 32 → 1
Loss Function: Binary Cross-Entropy

Test Set Metrics:
  AUC-ROC:     0.8879
  AUC-PR:      0.8301
  Brier Score: 0.0893



## 2. Model Comparison

MLP vs XGBoost performance on the test set.

In [ ]:
try:
    with open('results/xgboost_metrics.json', 'r') as f:
        xgb_metrics = json.load(f)

    xgb_test = xgb_metrics['test_metrics']
    mlp_test = metrics['test_metrics']

    print("=" * 70)
    print("MODEL COMPARISON (MLP vs XGBoost, Test Set)")
    print("=" * 70)
    print(f"\n{'Metric':<20} {'MLP':>10} {'XGBoost':>10}")
    print("-" * 42)
    print(f"{'AUC-ROC':<20} {mlp_test['auc_roc']:>10.4f} {xgb_test['auc_roc']:>10.4f}")
    print(f"{'AUC-PR':<20} {mlp_test['auc_pr']:>10.4f} {xgb_test['auc_pr']:>10.4f}")
    print(f"{'Brier Score':<20} {mlp_test['brier_score']:>10.4f} {xgb_test['brier_score']:>10.4f}")
    if 'calibration_gap' in mlp_test and 'calibration_gap' in xgb_test:
        print(f"{'Calibration Gap':<20} {mlp_test['calibration_gap']:>10.4f} {xgb_test['calibration_gap']:>10.4f}")

    print("\n" + "=" * 70)

except FileNotFoundError:
    print("XGBoost metrics not found. Run notebooks/xgboost_training.ipynb to generate.")

# Previous: loaded top_features.csv (logistic regression coefficients — removed)
# features = pd.read_csv('results/top_features.csv')

## 3. Bias and Fairness Analysis

In [4]:
try:
    with open('results/bias_analysis.json', 'r') as f:
        bias = json.load(f)
    
    gender_df = pd.DataFrame(bias['gender_metrics'])
    age_df = pd.DataFrame(bias['age_metrics'])
    region_df = pd.DataFrame(bias['region_metrics'])
    
    print(f"Gender: Max calibration gap {gender_df['Calibration Gap'].max():.1%} across {len(gender_df)} groups")
    print(f"Age: Max calibration gap {age_df['Calibration Gap'].max():.1%} across {len(age_df)} groups")
    print(f"Regional: {len(region_df)} regions analyzed, AUC range {region_df['AUC'].min():.3f}-{region_df['AUC'].max():.3f}")
    
except FileNotFoundError:
    print("Bias analysis results not found. Run notebooks/bias_fairness_analysis.ipynb")

Gender: Max calibration gap 0.4% across 4 groups
Age: Max calibration gap 1.3% across 7 groups
Regional: 4 regions analyzed, AUC range 0.871-0.891


### Counterfactual Design Choices

credit_score and income are excluded from counterfactual generation. Both are known to influence approval decisions, but neither can be changed meaningfully in the short term by an applicant. The analysis focuses on loan structure variables (ltv, dtir1, term, loan_amount, property_value) that applicants can adjust during the application process.


In [5]:
try:
    cf_summary = pd.read_csv('results/dice_counterfactuals/verification_summary.csv')
    
    print("=" * 70)
    print("COUNTERFACTUAL EXPLANATIONS")
    print("=" * 70)
    
    print(f"\nCases analyzed: {len(cf_summary)}")
    print(f"Counterfactuals per case: 5")
    
    # Separate high-risk (original_pred=1) from low-risk (original_pred=0)
    high_risk = cf_summary[cf_summary['original_pred'] == 1]
    low_risk = cf_summary[cf_summary['original_pred'] == 0]
    
    print(f"\nHigh-risk cases (predicted default):")
    print(f"  Count: {len(high_risk)}")
    if len(high_risk) > 0:
        hr_flip_rate = high_risk['flip_rate'].mean()
        hr_flipped = (high_risk['flip_rate'] > 0).sum()
        print(f"  Cases with successful counterfactuals: {hr_flipped}/{len(high_risk)} ({hr_flipped/len(high_risk):.0%})")
        print(f"  Average flip rate: {hr_flip_rate:.1%}")
    
    print(f"\nLow-risk cases (predicted no default):")
    print(f"  Count: {len(low_risk)}")
    if len(low_risk) > 0:
        lr_flip_rate = low_risk['flip_rate'].mean()
        print(f"  Average flip rate: {lr_flip_rate:.1%}")
        print(f"  (Low flip rate expected - already approved)")
    
    overall_flip = cf_summary['flip_rate'].mean()
    print(f"\nOverall flip rate: {overall_flip:.1%}")
    
    # Show detailed summary
    print("\nDetailed Summary:")
    display_cols = ['case_index', 'original_proba', 'original_pred', 'num_counterfactuals', 'num_flipped', 'flip_rate']
    print(cf_summary[display_cols].to_string(index=False))
    
    print("\n" + "=" * 70)
    
except FileNotFoundError:
    print("Counterfactual results not found.")
    print("Run notebooks/generate_counterfactuals.ipynb to generate.")

COUNTERFACTUAL EXPLANATIONS

Cases analyzed: 8
Counterfactuals per case: 5

High-risk cases (predicted default):
  Count: 4
  Cases with successful counterfactuals: 4/4 (100%)
  Average flip rate: 100.0%

Low-risk cases (predicted no default):
  Count: 4
  Average flip rate: 0.0%
  (Low flip rate expected - already approved)

Overall flip rate: 50.0%

Detailed Summary:
 case_index  original_proba  original_pred  num_counterfactuals  num_flipped  flip_rate
       6183        0.524392              1                    5            5        1.0
      12844        0.278831              0                    5            0        0.0
      14795        0.134809              0                    5            0        0.0
       7433        0.945600              1                    5            5        1.0
      12543        0.569181              1                    5            5        1.0
      11660        0.818894              1                    5            5        1.0
       4680 

### Counterfactual Design Choices

credit_score and income are excluded from counterfactual generation. Both are known to influence approval decisions, but neither can be changed meaningfully in the short term by an applicant. The analysis focuses on loan structure variables (ltv, dtir1, term, loan_amount, property_value) that applicants can adjust during the application process.


In [6]:
try:
    with open('results/dice_counterfactuals/summary_statistics.json', 'r') as f:
        stats = json.load(f)
    
    avg_features = stats['features_changed']['average']
    feature_counts = stats['most_commonly_changed_features']
    total_cfs = stats['total_counterfactuals_generated']
    
    print(f"Minimal changes: {avg_features:.1f} features on average")
    print("\nMost effective actions:")
    
    actionable = ['ltv', 'dtir1', 'term', 'property_value', 'loan_amount']
    for feat in actionable:
        if feat in feature_counts:
            pct = (feature_counts[feat] / total_cfs) * 100
            feat_name = {'ltv': 'Reduce LTV (increase down payment)', 
                        'dtir1': 'Reduce DTIR (pay down debt)',
                        'term': 'Adjust loan term',
                        'property_value': 'Choose less expensive property',
                        'loan_amount': 'Request smaller loan'}.get(feat, feat)
            print(f"  - {feat_name}: {pct:.0f}% of counterfactuals")
    
except FileNotFoundError:
    print("Run notebooks/counterfactual_summary_statistics.ipynb to generate statistics")

Minimal changes: 2.2 features on average

Most effective actions:
  - Reduce LTV (increase down payment): 59% of counterfactuals
  - Reduce DTIR (pay down debt): 57% of counterfactuals
  - Adjust loan term: 54% of counterfactuals
  - Choose less expensive property: 30% of counterfactuals
  - Request smaller loan: 22% of counterfactuals


## 5. Interactive Prediction Demo

Load the model and make a sample prediction.

In [ ]:
# Load model architecture
class CreditMLP(nn.Module):
    def __init__(self, input_dim):
        super(CreditMLP, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(32, 1)
            # Sigmoid removed: BCEWithLogitsLoss applies it internally during training.
            # Previous: nn.Sigmoid()
        )

    def forward(self, x):
        return self.network(x)

checkpoint = torch.load('models/mlp_model.pth', map_location='cpu', weights_only=False)
model = CreditMLP(checkpoint['input_dim'])
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"Model loaded: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Example prediction on test data
test = pd.read_csv('data/test.csv')
sample = test.iloc[0:1].drop(columns=['status']).values

with torch.no_grad():
    logit = model(torch.FloatTensor(sample))
    pred_proba = torch.sigmoid(logit).numpy()[0, 0]

print("=" * 70)
print("EXAMPLE PREDICTION")
print("=" * 70)
print(f"\nSample from test set (index 0)")
print(f"Predicted probability: {pred_proba:.1%}")
print(f"\nDecision: {'REJECTED (high default risk)' if pred_proba >= 0.5 else 'APPROVED (low default risk)'}")
print(f"Actual label: {test.iloc[0]['status']} (0=no default, 1=default)")
print("\n" + "=" * 70)

print("\nTo make predictions on custom data:")
print("1. Prepare features in the same format as the test set")
print("2. Apply torch.sigmoid() to model output for probability predictions")

## Key Files

Data: data/train.csv, data/val.csv, data/test.csv
Models: models/mlp_model.pth, models/xgboost_model.json, models/preprocessor.pkl, models/training_meta.json
Results: results/mlp_predictions.csv, results/xgboost_predictions.csv, results/dice_counterfactuals/